# Evals

Evaluations, often called *evals*, test system outputs against criteria you specify. An eval can measure the accuracy of a calculation, the relevance of an explanation or compliance with a requested style. It needs example tasks, a rule for awarding credit and a way to apply that rule to the outputs.

Imagine comparing two retrieval-augmented generation (RAG) systems built on the same economics readings. Give both the same questions. For each reply, a grader can check whether the economic claim is correct, whether it addresses the question, and whether the cited passage supports it. If the reading only identifies a mechanism, a numerical estimate invented by the assistant should fail the source-support check.

This notebook uses a simpler application. I used an available economics textbook to select concepts, wrote short questions and designed item-specific scoring rubrics. You will grade two teaching answers, inspect the instructions for an LLM judge and calculate point shares from three historical model runs.

**To run the example**, open this notebook in Jupyter or VS Code, select a Python 3 kernel and choose **Run All**. It runs offline from the included task, judge instructions and numerical marks. You can change your grades and rerun the comparison. Live generation and grading were performed in the original evaluation.

## 1. Define the task

What would convince you that a model understands efficiency in a two-good exchange economy? You might want it to state the marginal condition, explain its geometry and connect it to incentives to trade. Asking “What is Pareto efficiency?” could elicit a correct definition while revealing little about those three abilities.

Begin with a question that directly elicits the reasoning you intend to assess. The teaching question below paraphrases one item from the general equilibrium and welfare bank. Each candidate gets the same question and requested answer format. Keep track of the model version, run date, instructions, permitted resources and generation settings so you can interpret differences later.

In [ ]:
question = (
    "Consider a smooth two-good exchange economy with two consumers at an interior "
    "Pareto-efficient allocation. How do their marginal trade-offs compare? "
    "Explain the geometry and why the condition matters for trade."
)
answer_format = "Use three bullets with one claim each, and only the bullets."

def candidate_prompt(question, answer_format):
    return f"Answer the economics question in the requested form.\n\n{question}\n{answer_format}"

print(candidate_prompt(question, answer_format))

## 2. Design the rubric

A rubric says which features of an answer earn points. Draft an ideal answer first, then break it into claims that two graders can check independently. For this smooth interior allocation, equal marginal willingness to trade gives the two indifference curves a common tangent. If the two rates differed, a small trade at a rate between them could benefit both consumers.

| One point for each | Evidence sufficient for credit | A borderline response |
| --- | --- | --- |
| **Marginal condition** | States that both consumers give up the same amount of one good for a marginal unit of the other. Equivalent notation earns credit. | “Demand is balanced” leaves the trade-off unspecified. |
| **Interior geometry** | Explains that the indifference curves share a tangent, or have the same slope at this interior allocation. | Simply repeating “interior allocation” leaves the geometry unexplained. |
| **Gains from exchange** | Connects efficiency to the absence of a feasible mutually beneficial trade, or explains how unequal rates would allow one. | “The allocation is efficient” repeats the premise. |

These are teaching anchors for the historical item's three economic ideas. In particular, the exercise requires an explicit geometric explanation for the second point. The historical criterion named the interior case and tangency more briefly. A decision to tighten an anchor changes the scoring rule and merits a new rubric version.

Keep the format flag separate from the three content points. An answer can meet all three economic criteria while overlooking the request for bullets. Before grading a bank, try the rubric on a clear, a partial and a borderline answer. If two economists disagree, identify which row is ambiguous and revise it before scoring the comparison.

In [ ]:
rubric = [
    "States equal marginal willingness to trade across consumers, in words or notation.",
    "Explains the shared tangent or equal slopes of indifference curves at the interior allocation.",
    "Explains why no feasible trade can benefit both, or why unequal rates would permit one.",
]

teaching_answers = {
    "A": (
        "- Both households give up the same quantity of good 2 for a marginal unit of good 1.\n"
        "- If their rates differed, they could exchange at a rate in between and both gain.\n"
        "- Efficiency rules out such a trade."
    ),
    "B": (
        "At the interior allocation the two indifference curves share a tangent. "
        "Both households assign the same marginal value to good 1 in units of good 2. "
        "A small feasible exchange can no longer benefit both."
    ),
}
for label, answer in teaching_answers.items():
    print(f"Answer {label}\n{answer}\n")

## 3. Grade the answers

**Try grading A and B before looking at the key.** The two answers above were written for this exercise. Enter your three 0/1 marks in rubric order, and record whether the requested structure was violated. For example, an entry has the shape `([0, 0, 0], False)`. Leave a value as `None` to see the suggested key without submitting marks.

A human grader can highlight the sentence that supports each mark. An LLM judge can follow the same procedure at scale. In both cases, credit depends on the rubric and on the grader's interpretation of the answer.

In [ ]:
your_grades = {
    "A": None,
    "B": None,
}
teaching_key = {
    "A": ([1, 0, 1], False),
    "B": ([1, 1, 1], True),
}
explanations = {
    "A": "The equal trade-off and the trade between unequal rates are explicit. The geometry is missing.",
    "B": "The common tangent, equal trade-off and absence of mutual gain are all explicit. The answer is one paragraph.",
}
for label, (key_marks, key_flag) in teaching_key.items():
    print(f"{label} | suggested marks {key_marks} | {sum(key_marks)}/3 | format violation {key_flag}")
    print(explanations[label])
    if your_grades[label] is None:
        continue
    marks, flag = your_grades[label]
    if len(marks) != len(rubric) or any(type(mark) is not int or mark not in (0, 1) for mark in marks):
        raise ValueError("Enter one integer, 0 or 1, for each rubric row.")
    if type(flag) is not bool:
        raise ValueError("Use True or False for the format violation.")
    differences = [i for i, (yours, key) in enumerate(zip(marks, key_marks), 1) if yours != key]
    print(f"Your total is {sum(marks)}/3. Rubric rows to discuss: {differences}. Format agreement: {flag == key_flag}.")

**What did the LLM judge see?** The original runner passed the grading instructions, item ID, question, answer format, rubric and candidate's answer. The researcher also wrote an ideal answer for the item. The judge input omitted it and the candidate's identity. Candidates saw the answering instructions, question and format. The model being evaluated had no access to the grader's checklist through that prompt.

The next cell stores the project’s full judge instructions, including its two generic grading examples. The following cell builds a reusable message with those instructions and the teaching answer of your choice. It prints the inputs for inspection. A human could use the same checklist on paper.

In [ ]:
# Original grading instructions, including two scoring examples.
JUDGE_INSTRUCTIONS = r"""You are grading an economics evaluation answer using a checklist rubric.

You will be given:
- id (integer)
- question (string)
- answer_format (string)
- rubric (list of strings)
- model_answer (string)

Your job:
1) Score each rubric item as 1 if the model_answer clearly satisfies it, otherwise 0.
2) Be strict: if an item is only implied vaguely or is ambiguous, score 0.
3) Do not award partial credit: each item is binary.
4) Format violations are ONLY about structure, not content quality. Set format_violation=true ONLY if:
   - Wrong number of bullet points (e.g., 3 bullets when 4 required)
   - Added preamble text before the bullets (e.g., "Here is my answer:")
   - Added conclusion text after the bullets
   - Did not use bullet format when required (accepts *, -, •, or numbered lists as valid bullets)
   - For single-word/number answers: added extra words or explanation
   Do NOT flag as format violation: vague content, redundant bullets, wrong order, or content quality issues. Those affect rubric scores, not format.
5) If the model refuses to answer or says "I don't know", score all rubric items as 0.

Output ONLY valid JSON with exactly these keys:
- id: integer
- scores: array of integers (0/1), same length and order as rubric
- total: integer (sum of scores)
- format_violation: boolean
- reasoning: string (brief explanation for each score, e.g., "Item 1: stated X. Item 2: missing Y.")

---

Example 1:

Input:
- id: 1
- question: "Explain why a binding price ceiling causes a shortage."
- answer_format: "Answer in exactly 2 bullet points."
- rubric: ["States the ceiling is below equilibrium price", "Explains shortage as quantity demanded exceeding quantity supplied"]
- model_answer: "* A price ceiling is set below the market equilibrium price.\n* At this lower price, buyers want more but sellers supply less, creating a shortage."

Output:
{"id": 1, "scores": [1, 1], "total": 2, "format_violation": false, "reasoning": "Item 1: explicitly states ceiling is below equilibrium. Item 2: explains shortage via demand exceeding supply."}

---

Example 2:

Input:
- id: 2
- question: "What is the approximate percent change in quantity demanded if elasticity is -0.5 and price rises 10%?"
- answer_format: "Return only a single number with a percent sign."
- rubric: ["Correct answer: -5%"]
- model_answer: "The quantity demanded will fall by approximately -5%."

Output:
{"id": 2, "scores": [1], "total": 1, "format_violation": true, "reasoning": "Item 1: correct value -5%. Format violation: added explanatory text instead of just the number."}"""

In [ ]:
import json

def build_judge_prompt(item_id, question, answer_format, rubric, answer):
    fields = {
        "id": item_id,
        "question": question,
        "answer_format": answer_format,
        "rubric": rubric,
        "model_answer": answer,
    }
    inputs = "\n".join(f"- {name}: {json.dumps(value, ensure_ascii=False)}" for name, value in fields.items())
    return JUDGE_INSTRUCTIONS + "\n\n" + inputs

example_answer = "A"  # Change to "B" to inspect the other judge request.
message = build_judge_prompt(10, question, answer_format, rubric, teaching_answers[example_answer])
print(message)

**What did the original judge award?** Gemini 2.0 Flash graded the three archived runs used here. Its instructions call for one binary mark per criterion, a separate structure flag and a brief explanation. They give a vague implication zero and assign zero to all criteria for an explicit refusal. A missing or failed grading call is an operational issue, which we treat separately in step 4.

The next cells replay the recorded marks for the original item. These belong to the historical candidates and their original rubric. A and B above belong to the teaching exercise. The embedded data contain item IDs, point budgets, topic and difficulty tags, model identifiers, recorded marks and format flags.

In [ ]:
BANK = [{'id': 1, 'topic': 'ge_exchange', 'difficulty': 1, 'max_points': 3}, {'id': 2, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 3, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 4, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 5, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 6, 'topic': 'ge_exchange', 'difficulty': 3, 'max_points': 3}, {'id': 7, 'topic': 'ge_exchange', 'difficulty': 3, 'max_points': 3}, {'id': 8, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 9, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 10, 'topic': 'ge_exchange', 'difficulty': 3, 'max_points': 3}, {'id': 11, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 12, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 13, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 14, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 15, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 16, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 17, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 18, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 19, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 20, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 21, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 22, 'topic': 'production', 'difficulty': 3, 'max_points': 3}, {'id': 23, 'topic': 'production', 'difficulty': 3, 'max_points': 3}, {'id': 24, 'topic': 'production', 'difficulty': 3, 'max_points': 3}, {'id': 25, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 26, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 27, 'topic': 'public', 'difficulty': 3, 'max_points': 3}, {'id': 28, 'topic': 'trade', 'difficulty': 2, 'max_points': 3}, {'id': 29, 'topic': 'trade', 'difficulty': 2, 'max_points': 3}, {'id': 30, 'topic': 'trade', 'difficulty': 3, 'max_points': 3}, {'id': 31, 'topic': 'trade', 'difficulty': 3, 'max_points': 3}, {'id': 32, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 33, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 34, 'topic': 'dynamic', 'difficulty': 3, 'max_points': 3}, {'id': 35, 'topic': 'asset_pricing', 'difficulty': 2, 'max_points': 3}, {'id': 36, 'topic': 'asset_pricing', 'difficulty': 2, 'max_points': 3}, {'id': 37, 'topic': 'asset_pricing', 'difficulty': 3, 'max_points': 3}, {'id': 38, 'topic': 'asset_pricing', 'difficulty': 2, 'max_points': 3}, {'id': 39, 'topic': 'asset_pricing', 'difficulty': 2, 'max_points': 3}, {'id': 40, 'topic': 'incomplete_markets', 'difficulty': 2, 'max_points': 3}, {'id': 41, 'topic': 'incomplete_markets', 'difficulty': 3, 'max_points': 3}, {'id': 42, 'topic': 'incomplete_markets', 'difficulty': 3, 'max_points': 3}, {'id': 43, 'topic': 'firms', 'difficulty': 2, 'max_points': 3}, {'id': 44, 'topic': 'firms', 'difficulty': 3, 'max_points': 3}, {'id': 45, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 46, 'topic': 'ge_exchange', 'difficulty': 3, 'max_points': 3}, {'id': 47, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 48, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 49, 'topic': 'ge_exchange', 'difficulty': 3, 'max_points': 3}, {'id': 50, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 51, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 52, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 53, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 54, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 55, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 56, 'topic': 'production', 'difficulty': 3, 'max_points': 3}, {'id': 57, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 58, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 59, 'topic': 'production', 'difficulty': 3, 'max_points': 3}, {'id': 60, 'topic': 'trade', 'difficulty': 3, 'max_points': 3}, {'id': 61, 'topic': 'trade', 'difficulty': 3, 'max_points': 3}, {'id': 62, 'topic': 'trade', 'difficulty': 2, 'max_points': 3}, {'id': 63, 'topic': 'trade', 'difficulty': 2, 'max_points': 3}, {'id': 64, 'topic': 'trade', 'difficulty': 3, 'max_points': 3}, {'id': 65, 'topic': 'dynamic', 'difficulty': 3, 'max_points': 3}, {'id': 66, 'topic': 'dynamic', 'difficulty': 3, 'max_points': 3}, {'id': 67, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 68, 'topic': 'dynamic', 'difficulty': 3, 'max_points': 3}, {'id': 69, 'topic': 'dynamic', 'difficulty': 3, 'max_points': 3}, {'id': 70, 'topic': 'asset_pricing', 'difficulty': 3, 'max_points': 3}, {'id': 71, 'topic': 'asset_pricing', 'difficulty': 3, 'max_points': 3}, {'id': 72, 'topic': 'asset_pricing', 'difficulty': 2, 'max_points': 3}, {'id': 73, 'topic': 'asset_pricing', 'difficulty': 2, 'max_points': 3}, {'id': 74, 'topic': 'asset_pricing', 'difficulty': 3, 'max_points': 3}, {'id': 75, 'topic': 'incomplete_markets', 'difficulty': 3, 'max_points': 3}, {'id': 76, 'topic': 'incomplete_markets', 'difficulty': 3, 'max_points': 3}, {'id': 77, 'topic': 'incomplete_markets', 'difficulty': 2, 'max_points': 3}, {'id': 78, 'topic': 'incomplete_markets', 'difficulty': 2, 'max_points': 3}, {'id': 79, 'topic': 'incomplete_markets', 'difficulty': 3, 'max_points': 3}, {'id': 80, 'topic': 'firms', 'difficulty': 2, 'max_points': 3}, {'id': 81, 'topic': 'firms', 'difficulty': 3, 'max_points': 3}, {'id': 82, 'topic': 'firms', 'difficulty': 2, 'max_points': 3}, {'id': 83, 'topic': 'ge_exchange', 'difficulty': 3, 'max_points': 3}, {'id': 84, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 85, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 86, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 87, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 88, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 89, 'topic': 'production', 'difficulty': 3, 'max_points': 3}, {'id': 90, 'topic': 'trade', 'difficulty': 2, 'max_points': 3}, {'id': 91, 'topic': 'trade', 'difficulty': 3, 'max_points': 3}, {'id': 92, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 93, 'topic': 'dynamic', 'difficulty': 3, 'max_points': 3}, {'id': 94, 'topic': 'asset_pricing', 'difficulty': 2, 'max_points': 3}, {'id': 95, 'topic': 'asset_pricing', 'difficulty': 3, 'max_points': 3}, {'id': 96, 'topic': 'incomplete_markets', 'difficulty': 2, 'max_points': 3}, {'id': 97, 'topic': 'incomplete_markets', 'difficulty': 3, 'max_points': 3}, {'id': 98, 'topic': 'firms', 'difficulty': 3, 'max_points': 3}, {'id': 99, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 100, 'topic': 'ge_exchange', 'difficulty': 3, 'max_points': 3}, {'id': 101, 'topic': 'ge_exchange', 'difficulty': 1, 'max_points': 3}, {'id': 102, 'topic': 'ge_exchange', 'difficulty': 1, 'max_points': 3}, {'id': 103, 'topic': 'ge_exchange', 'difficulty': 1, 'max_points': 3}, {'id': 104, 'topic': 'ge_exchange', 'difficulty': 1, 'max_points': 3}, {'id': 105, 'topic': 'ge_exchange', 'difficulty': 1, 'max_points': 3}, {'id': 106, 'topic': 'ge_exchange', 'difficulty': 1, 'max_points': 3}, {'id': 107, 'topic': 'ge_exchange', 'difficulty': 1, 'max_points': 3}, {'id': 108, 'topic': 'ge_exchange', 'difficulty': 1, 'max_points': 3}, {'id': 109, 'topic': 'ge_exchange', 'difficulty': 1, 'max_points': 3}, {'id': 110, 'topic': 'ge_exchange', 'difficulty': 1, 'max_points': 3}, {'id': 111, 'topic': 'ge_exchange', 'difficulty': 1, 'max_points': 3}, {'id': 112, 'topic': 'ge_exchange', 'difficulty': 1, 'max_points': 3}, {'id': 113, 'topic': 'ge_exchange', 'difficulty': 1, 'max_points': 3}, {'id': 114, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 115, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 116, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 117, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 118, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 119, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 120, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 121, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 122, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 123, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 124, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 125, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 126, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 127, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 128, 'topic': 'ge_exchange', 'difficulty': 2, 'max_points': 3}, {'id': 129, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 130, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 131, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 132, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 133, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 134, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 135, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 136, 'topic': 'production', 'difficulty': 2, 'max_points': 3}, {'id': 137, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 138, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 139, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 140, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 141, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 142, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 143, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 144, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 145, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 146, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 147, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 148, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 149, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 150, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 151, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 152, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 153, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 154, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 155, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 156, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 157, 'topic': 'welfare', 'difficulty': 3, 'max_points': 3}, {'id': 158, 'topic': 'firms', 'difficulty': 2, 'max_points': 3}, {'id': 159, 'topic': 'firms', 'difficulty': 2, 'max_points': 3}, {'id': 160, 'topic': 'firms', 'difficulty': 2, 'max_points': 3}, {'id': 161, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 162, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 163, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 164, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 165, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 166, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 167, 'topic': 'dynamic', 'difficulty': 2, 'max_points': 3}, {'id': 168, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 169, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 170, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 171, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 172, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 173, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 174, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}, {'id': 175, 'topic': 'welfare', 'difficulty': 2, 'max_points': 3}]
RUNS = [{'model': 'gemini-3-pro-preview', 'judge': 'gemini-2.0-flash', 'date': '20260128', 'marks': [{'id': 1, 'scores': [1, 1, 0], 'format': False}, {'id': 2, 'scores': [1, 1, 1], 'format': False}, {'id': 3, 'scores': [1, 1, 1], 'format': False}, {'id': 4, 'scores': [1, 1, 1], 'format': False}, {'id': 5, 'scores': [1, 1, 1], 'format': False}, {'id': 6, 'scores': [1, 1, 0], 'format': False}, {'id': 7, 'scores': [1, 1, 1], 'format': False}, {'id': 8, 'scores': [1, 1, 1], 'format': False}, {'id': 9, 'scores': [0, 1, 0], 'format': False}, {'id': 10, 'scores': [1, 1, 0], 'format': False}, {'id': 11, 'scores': [1, 1, 1], 'format': False}, {'id': 12, 'scores': [1, 1, 1], 'format': False}, {'id': 13, 'scores': [1, 1, 1], 'format': False}, {'id': 14, 'scores': [1, 1, 1], 'format': False}, {'id': 15, 'scores': [1, 1, 1], 'format': False}, {'id': 16, 'scores': [1, 1, 1], 'format': False}, {'id': 17, 'scores': [1, 0, 1], 'format': False}, {'id': 18, 'scores': [1, 1, 0], 'format': False}, {'id': 19, 'scores': [1, 1, 1], 'format': False}, {'id': 20, 'scores': [0, 0, 1], 'format': False}, {'id': 21, 'scores': [1, 1, 1], 'format': False}, {'id': 22, 'scores': [1, 1, 1], 'format': False}, {'id': 23, 'scores': [1, 1, 1], 'format': False}, {'id': 24, 'scores': [1, 0, 1], 'format': False}, {'id': 25, 'scores': [1, 1, 0], 'format': False}, {'id': 26, 'scores': [1, 1, 1], 'format': False}, {'id': 27, 'scores': [1, 1, 1], 'format': False}, {'id': 28, 'scores': [1, 1, 1], 'format': False}, {'id': 29, 'scores': [0, 0, 0], 'format': False}, {'id': 30, 'scores': [1, 1, 1], 'format': False}, {'id': 31, 'scores': [1, 1, 1], 'format': False}, {'id': 32, 'scores': [1, 1, 1], 'format': False}, {'id': 33, 'scores': [1, 1, 0], 'format': False}, {'id': 34, 'scores': [1, 1, 1], 'format': False}, {'id': 35, 'scores': [1, 1, 1], 'format': False}, {'id': 36, 'scores': [1, 1, 1], 'format': False}, {'id': 37, 'scores': [1, 1, 0], 'format': False}, {'id': 38, 'scores': [1, 0, 1], 'format': False}, {'id': 39, 'scores': [1, 1, 1], 'format': False}, {'id': 40, 'scores': [1, 1, 0], 'format': False}, {'id': 41, 'scores': [1, 1, 1], 'format': False}, {'id': 42, 'scores': [1, 1, 1], 'format': False}, {'id': 43, 'scores': [1, 1, 1], 'format': False}, {'id': 44, 'scores': [1, 0, 1], 'format': False}, {'id': 45, 'scores': [1, 1, 1], 'format': False}, {'id': 46, 'scores': [1, 0, 0], 'format': False}, {'id': 47, 'scores': [1, 0, 1], 'format': False}, {'id': 48, 'scores': [1, 0, 1], 'format': False}, {'id': 49, 'scores': [1, 1, 1], 'format': False}, {'id': 50, 'scores': [1, 1, 1], 'format': False}, {'id': 51, 'scores': [1, 1, 1], 'format': False}, {'id': 52, 'scores': [1, 1, 1], 'format': False}, {'id': 53, 'scores': [1, 1, 1], 'format': False}, {'id': 54, 'scores': [1, 1, 1], 'format': False}, {'id': 55, 'scores': [1, 1, 1], 'format': False}, {'id': 56, 'scores': [1, 1, 1], 'format': False}, {'id': 57, 'scores': [1, 1, 1], 'format': False}, {'id': 58, 'scores': [1, 1, 1], 'format': False}, {'id': 59, 'scores': [1, 1, 1], 'format': False}, {'id': 60, 'scores': [1, 1, 1], 'format': False}, {'id': 61, 'scores': [1, 1, 1], 'format': False}, {'id': 62, 'scores': [0, 1, 1], 'format': True}, {'id': 63, 'scores': [1, 1, 1], 'format': False}, {'id': 64, 'scores': [1, 1, 1], 'format': False}, {'id': 65, 'scores': [1, 1, 1], 'format': False}, {'id': 66, 'scores': [1, 1, 1], 'format': False}, {'id': 67, 'scores': [0, 0, 0], 'format': False}, {'id': 68, 'scores': [1, 1, 1], 'format': False}, {'id': 69, 'scores': [1, 1, 1], 'format': False}, {'id': 70, 'scores': [0, 0, 0], 'format': True}, {'id': 71, 'scores': [1, 1, 0], 'format': False}, {'id': 72, 'scores': [1, 1, 1], 'format': False}, {'id': 73, 'scores': [1, 1, 1], 'format': False}, {'id': 74, 'scores': [1, 1, 1], 'format': False}, {'id': 75, 'scores': None, 'format': None}, {'id': 76, 'scores': [1, 1, 1], 'format': False}, {'id': 77, 'scores': [1, 1, 1], 'format': False}, {'id': 78, 'scores': [1, 1, 1], 'format': False}, {'id': 79, 'scores': [1, 1, 1], 'format': False}, {'id': 80, 'scores': [1, 1, 1], 'format': False}, {'id': 81, 'scores': [1, 1, 1], 'format': False}, {'id': 82, 'scores': [1, 1, 1], 'format': False}, {'id': 83, 'scores': [1, 1, 1], 'format': False}, {'id': 84, 'scores': [1, 0, 0], 'format': False}, {'id': 85, 'scores': [1, 1, 1], 'format': False}, {'id': 86, 'scores': [1, 1, 1], 'format': False}, {'id': 87, 'scores': [1, 1, 1], 'format': False}, {'id': 88, 'scores': [1, 1, 1], 'format': False}, {'id': 89, 'scores': [1, 1, 1], 'format': False}, {'id': 90, 'scores': [1, 1, 1], 'format': False}, {'id': 91, 'scores': [1, 1, 1], 'format': False}, {'id': 92, 'scores': [1, 1, 1], 'format': False}, {'id': 93, 'scores': [1, 1, 1], 'format': False}, {'id': 94, 'scores': [1, 1, 1], 'format': False}, {'id': 95, 'scores': [1, 1, 1], 'format': False}, {'id': 96, 'scores': [1, 1, 1], 'format': False}, {'id': 97, 'scores': [1, 1, 1], 'format': False}, {'id': 98, 'scores': [1, 1, 0], 'format': False}, {'id': 99, 'scores': [1, 1, 1], 'format': False}, {'id': 100, 'scores': [1, 1, 1], 'format': False}, {'id': 101, 'scores': [1, 1, 1], 'format': False}, {'id': 102, 'scores': [0, 0, 0], 'format': False}, {'id': 103, 'scores': [1, 1, 1], 'format': False}, {'id': 104, 'scores': [0, 1, 1], 'format': False}, {'id': 105, 'scores': [1, 1, 1], 'format': False}, {'id': 106, 'scores': [1, 1, 1], 'format': False}, {'id': 107, 'scores': [1, 1, 1], 'format': False}, {'id': 108, 'scores': [0, 0, 0], 'format': False}, {'id': 109, 'scores': [1, 1, 1], 'format': False}, {'id': 110, 'scores': [1, 1, 1], 'format': False}, {'id': 111, 'scores': [1, 1, 1], 'format': False}, {'id': 112, 'scores': [1, 0, 0], 'format': False}, {'id': 113, 'scores': [1, 0, 1], 'format': False}, {'id': 114, 'scores': [1, 1, 0], 'format': False}, {'id': 115, 'scores': [1, 1, 1], 'format': False}, {'id': 116, 'scores': [1, 1, 1], 'format': False}, {'id': 117, 'scores': [1, 1, 1], 'format': False}, {'id': 118, 'scores': [1, 1, 1], 'format': False}, {'id': 119, 'scores': [1, 1, 1], 'format': False}, {'id': 120, 'scores': [1, 1, 1], 'format': False}, {'id': 121, 'scores': [1, 1, 1], 'format': False}, {'id': 122, 'scores': [1, 1, 1], 'format': False}, {'id': 123, 'scores': [1, 1, 1], 'format': False}, {'id': 124, 'scores': [1, 1, 0], 'format': False}, {'id': 125, 'scores': [1, 1, 1], 'format': False}, {'id': 126, 'scores': [1, 0, 1], 'format': False}, {'id': 127, 'scores': [1, 1, 1], 'format': False}, {'id': 128, 'scores': [1, 0, 0], 'format': False}, {'id': 129, 'scores': [0, 0, 0], 'format': True}, {'id': 130, 'scores': [1, 1, 0], 'format': False}, {'id': 131, 'scores': [0, 0, 1], 'format': False}, {'id': 132, 'scores': [1, 1, 1], 'format': False}, {'id': 133, 'scores': [1, 1, 1], 'format': False}, {'id': 134, 'scores': [0, 0, 0], 'format': False}, {'id': 135, 'scores': [1, 1, 1], 'format': False}, {'id': 136, 'scores': [1, 1, 1], 'format': False}, {'id': 137, 'scores': [1, 1, 0], 'format': False}, {'id': 138, 'scores': [1, 0, 1], 'format': False}, {'id': 139, 'scores': [1, 1, 1], 'format': False}, {'id': 140, 'scores': [1, 1, 1], 'format': False}, {'id': 141, 'scores': [1, 1, 1], 'format': False}, {'id': 142, 'scores': [1, 1, 1], 'format': False}, {'id': 143, 'scores': [1, 1, 1], 'format': False}, {'id': 144, 'scores': [1, 1, 0], 'format': False}, {'id': 145, 'scores': [1, 1, 0], 'format': False}, {'id': 146, 'scores': [0, 0, 1], 'format': False}, {'id': 147, 'scores': [1, 1, 0], 'format': False}, {'id': 148, 'scores': [1, 1, 1], 'format': False}, {'id': 149, 'scores': [1, 0, 0], 'format': False}, {'id': 150, 'scores': [1, 1, 1], 'format': False}, {'id': 151, 'scores': [1, 1, 1], 'format': False}, {'id': 152, 'scores': [1, 1, 1], 'format': False}, {'id': 153, 'scores': [1, 1, 1], 'format': False}, {'id': 154, 'scores': [1, 1, 1], 'format': False}, {'id': 155, 'scores': [1, 1, 1], 'format': False}, {'id': 156, 'scores': [1, 1, 1], 'format': False}, {'id': 157, 'scores': [1, 1, 1], 'format': False}, {'id': 158, 'scores': [0, 1, 0], 'format': False}, {'id': 159, 'scores': [1, 1, 1], 'format': False}, {'id': 160, 'scores': [1, 1, 0], 'format': False}, {'id': 161, 'scores': [0, 0, 0], 'format': False}, {'id': 162, 'scores': [1, 1, 1], 'format': False}, {'id': 163, 'scores': [1, 1, 1], 'format': False}, {'id': 164, 'scores': [1, 1, 1], 'format': False}, {'id': 165, 'scores': [1, 1, 1], 'format': False}, {'id': 166, 'scores': [1, 1, 1], 'format': False}, {'id': 167, 'scores': [1, 1, 1], 'format': False}, {'id': 168, 'scores': [1, 0, 0], 'format': False}, {'id': 169, 'scores': [1, 1, 1], 'format': False}, {'id': 170, 'scores': [1, 1, 1], 'format': False}, {'id': 171, 'scores': [1, 1, 1], 'format': False}, {'id': 172, 'scores': [1, 1, 0], 'format': False}, {'id': 173, 'scores': [1, 1, 1], 'format': False}, {'id': 174, 'scores': [1, 1, 1], 'format': False}, {'id': 175, 'scores': [1, 0, 1], 'format': False}]}, {'model': 'gpt-5.2', 'judge': 'gemini-2.0-flash', 'date': '20260129', 'marks': [{'id': 1, 'scores': [1, 1, 0], 'format': False}, {'id': 2, 'scores': [1, 1, 1], 'format': False}, {'id': 3, 'scores': [1, 1, 1], 'format': False}, {'id': 4, 'scores': [0, 0, 0], 'format': False}, {'id': 5, 'scores': [1, 1, 1], 'format': False}, {'id': 6, 'scores': [1, 1, 0], 'format': False}, {'id': 7, 'scores': [1, 1, 1], 'format': False}, {'id': 8, 'scores': [1, 1, 1], 'format': False}, {'id': 9, 'scores': [1, 1, 0], 'format': False}, {'id': 10, 'scores': [1, 1, 0], 'format': False}, {'id': 11, 'scores': [1, 1, 1], 'format': False}, {'id': 12, 'scores': [1, 1, 1], 'format': False}, {'id': 13, 'scores': [1, 1, 1], 'format': False}, {'id': 14, 'scores': [1, 1, 1], 'format': False}, {'id': 15, 'scores': [1, 1, 1], 'format': False}, {'id': 16, 'scores': [1, 1, 1], 'format': False}, {'id': 17, 'scores': [1, 1, 1], 'format': False}, {'id': 18, 'scores': [1, 1, 0], 'format': False}, {'id': 19, 'scores': [1, 1, 1], 'format': False}, {'id': 20, 'scores': [1, 1, 1], 'format': False}, {'id': 21, 'scores': [1, 1, 1], 'format': False}, {'id': 22, 'scores': [1, 1, 1], 'format': False}, {'id': 23, 'scores': [1, 1, 1], 'format': False}, {'id': 24, 'scores': None, 'format': None}, {'id': 25, 'scores': [1, 1, 1], 'format': False}, {'id': 26, 'scores': [1, 1, 1], 'format': False}, {'id': 27, 'scores': [1, 1, 1], 'format': False}, {'id': 28, 'scores': [1, 1, 1], 'format': False}, {'id': 29, 'scores': [1, 0, 0], 'format': False}, {'id': 30, 'scores': [1, 1, 0], 'format': False}, {'id': 31, 'scores': [1, 1, 1], 'format': False}, {'id': 32, 'scores': [1, 1, 1], 'format': False}, {'id': 33, 'scores': [1, 1, 0], 'format': False}, {'id': 34, 'scores': [1, 1, 1], 'format': False}, {'id': 35, 'scores': [1, 1, 1], 'format': False}, {'id': 36, 'scores': [1, 1, 1], 'format': False}, {'id': 37, 'scores': [1, 0, 1], 'format': False}, {'id': 38, 'scores': [1, 0, 1], 'format': False}, {'id': 39, 'scores': [1, 1, 1], 'format': False}, {'id': 40, 'scores': [1, 1, 0], 'format': False}, {'id': 41, 'scores': [1, 1, 1], 'format': False}, {'id': 42, 'scores': [1, 1, 1], 'format': False}, {'id': 43, 'scores': [1, 1, 1], 'format': False}, {'id': 44, 'scores': [1, 1, 1], 'format': False}, {'id': 45, 'scores': [1, 1, 1], 'format': False}, {'id': 46, 'scores': [1, 1, 1], 'format': False}, {'id': 47, 'scores': [1, 1, 1], 'format': False}, {'id': 48, 'scores': [1, 1, 1], 'format': False}, {'id': 49, 'scores': [1, 1, 1], 'format': False}, {'id': 50, 'scores': [1, 1, 1], 'format': False}, {'id': 51, 'scores': [1, 1, 1], 'format': False}, {'id': 52, 'scores': [1, 0, 1], 'format': False}, {'id': 53, 'scores': [1, 1, 1], 'format': False}, {'id': 54, 'scores': [1, 1, 1], 'format': False}, {'id': 55, 'scores': [1, 1, 1], 'format': False}, {'id': 56, 'scores': [1, 1, 1], 'format': False}, {'id': 57, 'scores': [1, 1, 1], 'format': False}, {'id': 58, 'scores': [1, 1, 1], 'format': False}, {'id': 59, 'scores': [1, 1, 1], 'format': False}, {'id': 60, 'scores': [1, 1, 1], 'format': False}, {'id': 61, 'scores': [1, 1, 1], 'format': False}, {'id': 62, 'scores': [0, 1, 1], 'format': False}, {'id': 63, 'scores': [1, 1, 1], 'format': False}, {'id': 64, 'scores': [1, 1, 1], 'format': False}, {'id': 65, 'scores': [1, 1, 1], 'format': False}, {'id': 66, 'scores': None, 'format': None}, {'id': 67, 'scores': [1, 1, 0], 'format': False}, {'id': 68, 'scores': [1, 1, 1], 'format': False}, {'id': 69, 'scores': [1, 1, 1], 'format': False}, {'id': 70, 'scores': [0, 0, 0], 'format': True}, {'id': 71, 'scores': [1, 1, 1], 'format': False}, {'id': 72, 'scores': [1, 1, 1], 'format': False}, {'id': 73, 'scores': [1, 1, 1], 'format': False}, {'id': 74, 'scores': [1, 1, 1], 'format': False}, {'id': 75, 'scores': [1, 1, 1], 'format': False}, {'id': 76, 'scores': [1, 1, 1], 'format': False}, {'id': 77, 'scores': [1, 1, 1], 'format': False}, {'id': 78, 'scores': [1, 1, 1], 'format': False}, {'id': 79, 'scores': [1, 1, 1], 'format': False}, {'id': 80, 'scores': [1, 1, 1], 'format': False}, {'id': 81, 'scores': [1, 1, 1], 'format': False}, {'id': 82, 'scores': [1, 1, 1], 'format': False}, {'id': 83, 'scores': [1, 1, 1], 'format': False}, {'id': 84, 'scores': [1, 0, 0], 'format': False}, {'id': 85, 'scores': [1, 1, 1], 'format': False}, {'id': 86, 'scores': [1, 1, 1], 'format': False}, {'id': 87, 'scores': [1, 1, 1], 'format': False}, {'id': 88, 'scores': [1, 1, 1], 'format': False}, {'id': 89, 'scores': [1, 1, 1], 'format': False}, {'id': 90, 'scores': [1, 1, 1], 'format': False}, {'id': 91, 'scores': [1, 1, 1], 'format': False}, {'id': 92, 'scores': [1, 1, 1], 'format': False}, {'id': 93, 'scores': [1, 1, 0], 'format': False}, {'id': 94, 'scores': [1, 1, 1], 'format': False}, {'id': 95, 'scores': [1, 1, 0], 'format': False}, {'id': 96, 'scores': [1, 1, 1], 'format': False}, {'id': 97, 'scores': [1, 1, 1], 'format': False}, {'id': 98, 'scores': [1, 1, 1], 'format': False}, {'id': 99, 'scores': [1, 1, 1], 'format': False}, {'id': 100, 'scores': [1, 1, 1], 'format': False}, {'id': 101, 'scores': [1, 1, 1], 'format': False}, {'id': 102, 'scores': [1, 1, 1], 'format': False}, {'id': 103, 'scores': [1, 1, 1], 'format': False}, {'id': 104, 'scores': [1, 1, 1], 'format': False}, {'id': 105, 'scores': [1, 1, 1], 'format': False}, {'id': 106, 'scores': [0, 1, 0], 'format': False}, {'id': 107, 'scores': [1, 1, 1], 'format': False}, {'id': 108, 'scores': [1, 1, 0], 'format': False}, {'id': 109, 'scores': [1, 1, 1], 'format': False}, {'id': 110, 'scores': [1, 1, 1], 'format': False}, {'id': 111, 'scores': [1, 1, 1], 'format': False}, {'id': 112, 'scores': [1, 0, 0], 'format': False}, {'id': 113, 'scores': [1, 1, 1], 'format': False}, {'id': 114, 'scores': [1, 1, 0], 'format': False}, {'id': 115, 'scores': [1, 1, 1], 'format': False}, {'id': 116, 'scores': [1, 1, 1], 'format': False}, {'id': 117, 'scores': [1, 1, 1], 'format': False}, {'id': 118, 'scores': [1, 1, 1], 'format': False}, {'id': 119, 'scores': [1, 1, 1], 'format': False}, {'id': 120, 'scores': [1, 1, 1], 'format': False}, {'id': 121, 'scores': [1, 1, 1], 'format': False}, {'id': 122, 'scores': [1, 1, 1], 'format': False}, {'id': 123, 'scores': [1, 1, 1], 'format': False}, {'id': 124, 'scores': [1, 1, 1], 'format': False}, {'id': 125, 'scores': [1, 1, 1], 'format': False}, {'id': 126, 'scores': [1, 1, 1], 'format': False}, {'id': 127, 'scores': [1, 1, 1], 'format': False}, {'id': 128, 'scores': [1, 1, 1], 'format': False}, {'id': 129, 'scores': [1, 1, 1], 'format': False}, {'id': 130, 'scores': [1, 1, 0], 'format': False}, {'id': 131, 'scores': None, 'format': None}, {'id': 132, 'scores': [1, 1, 1], 'format': False}, {'id': 133, 'scores': [1, 1, 1], 'format': False}, {'id': 134, 'scores': [1, 1, 1], 'format': False}, {'id': 135, 'scores': [1, 1, 1], 'format': False}, {'id': 136, 'scores': [1, 1, 0], 'format': False}, {'id': 137, 'scores': [1, 1, 1], 'format': False}, {'id': 138, 'scores': [1, 1, 1], 'format': False}, {'id': 139, 'scores': [1, 1, 1], 'format': False}, {'id': 140, 'scores': [1, 1, 1], 'format': False}, {'id': 141, 'scores': None, 'format': None}, {'id': 142, 'scores': [1, 1, 1], 'format': False}, {'id': 143, 'scores': [1, 1, 1], 'format': False}, {'id': 144, 'scores': [1, 1, 0], 'format': False}, {'id': 145, 'scores': [1, 1, 0], 'format': False}, {'id': 146, 'scores': [0, 1, 0], 'format': False}, {'id': 147, 'scores': [1, 1, 1], 'format': False}, {'id': 148, 'scores': [1, 1, 1], 'format': False}, {'id': 149, 'scores': [1, 1, 1], 'format': False}, {'id': 150, 'scores': [1, 1, 1], 'format': False}, {'id': 151, 'scores': [1, 1, 1], 'format': False}, {'id': 152, 'scores': [1, 1, 1], 'format': False}, {'id': 153, 'scores': [1, 1, 1], 'format': False}, {'id': 154, 'scores': [1, 1, 1], 'format': False}, {'id': 155, 'scores': [1, 1, 1], 'format': False}, {'id': 156, 'scores': [1, 1, 1], 'format': False}, {'id': 157, 'scores': [1, 1, 1], 'format': False}, {'id': 158, 'scores': [1, 1, 0], 'format': False}, {'id': 159, 'scores': [1, 1, 1], 'format': False}, {'id': 160, 'scores': [0, 1, 0], 'format': False}, {'id': 161, 'scores': [0, 1, 1], 'format': False}, {'id': 162, 'scores': [1, 1, 0], 'format': False}, {'id': 163, 'scores': [1, 1, 1], 'format': False}, {'id': 164, 'scores': [1, 1, 1], 'format': False}, {'id': 165, 'scores': [1, 1, 0], 'format': False}, {'id': 166, 'scores': [1, 1, 1], 'format': False}, {'id': 167, 'scores': [0, 1, 1], 'format': False}, {'id': 168, 'scores': [1, 1, 1], 'format': False}, {'id': 169, 'scores': [1, 1, 1], 'format': False}, {'id': 170, 'scores': [1, 1, 1], 'format': False}, {'id': 171, 'scores': [1, 1, 1], 'format': False}, {'id': 172, 'scores': [1, 1, 1], 'format': False}, {'id': 173, 'scores': [1, 1, 1], 'format': False}, {'id': 174, 'scores': [1, 1, 1], 'format': False}, {'id': 175, 'scores': [1, 0, 1], 'format': False}]}, {'model': 'claude-opus-4-6', 'judge': 'gemini-2.0-flash', 'date': '20260212', 'marks': [{'id': 1, 'scores': None, 'format': None}, {'id': 2, 'scores': [1, 1, 1], 'format': False}, {'id': 3, 'scores': [1, 1, 1], 'format': False}, {'id': 4, 'scores': [1, 1, 1], 'format': False}, {'id': 5, 'scores': [1, 1, 1], 'format': False}, {'id': 6, 'scores': [1, 1, 0], 'format': False}, {'id': 7, 'scores': [1, 1, 1], 'format': False}, {'id': 8, 'scores': [1, 1, 1], 'format': False}, {'id': 9, 'scores': [0, 0, 0], 'format': False}, {'id': 10, 'scores': [1, 0, 1], 'format': False}, {'id': 11, 'scores': [1, 1, 1], 'format': False}, {'id': 12, 'scores': [1, 1, 1], 'format': False}, {'id': 13, 'scores': [1, 1, 1], 'format': False}, {'id': 14, 'scores': [1, 1, 1], 'format': False}, {'id': 15, 'scores': [1, 1, 0], 'format': False}, {'id': 16, 'scores': [1, 1, 1], 'format': False}, {'id': 17, 'scores': [1, 1, 1], 'format': False}, {'id': 18, 'scores': [1, 1, 0], 'format': False}, {'id': 19, 'scores': [1, 1, 1], 'format': False}, {'id': 20, 'scores': [1, 1, 1], 'format': False}, {'id': 21, 'scores': [1, 1, 1], 'format': False}, {'id': 22, 'scores': [1, 1, 1], 'format': False}, {'id': 23, 'scores': [1, 1, 1], 'format': False}, {'id': 24, 'scores': [1, 1, 1], 'format': False}, {'id': 25, 'scores': [1, 1, 1], 'format': False}, {'id': 26, 'scores': None, 'format': None}, {'id': 27, 'scores': [1, 1, 1], 'format': False}, {'id': 28, 'scores': [1, 1, 1], 'format': False}, {'id': 29, 'scores': [0, 0, 0], 'format': True}, {'id': 30, 'scores': [1, 1, 1], 'format': False}, {'id': 31, 'scores': [1, 1, 0], 'format': True}, {'id': 32, 'scores': None, 'format': None}, {'id': 33, 'scores': None, 'format': None}, {'id': 34, 'scores': [1, 1, 1], 'format': False}, {'id': 35, 'scores': [1, 1, 1], 'format': False}, {'id': 36, 'scores': [1, 1, 1], 'format': False}, {'id': 37, 'scores': [1, 1, 1], 'format': False}, {'id': 38, 'scores': [1, 1, 1], 'format': False}, {'id': 39, 'scores': [1, 1, 1], 'format': False}, {'id': 40, 'scores': [1, 1, 1], 'format': False}, {'id': 41, 'scores': [1, 1, 1], 'format': False}, {'id': 42, 'scores': None, 'format': None}, {'id': 43, 'scores': [1, 1, 1], 'format': False}, {'id': 44, 'scores': [1, 1, 1], 'format': False}, {'id': 45, 'scores': [1, 1, 1], 'format': False}, {'id': 46, 'scores': [1, 1, 1], 'format': False}, {'id': 47, 'scores': [1, 1, 0], 'format': False}, {'id': 48, 'scores': [1, 1, 1], 'format': False}, {'id': 49, 'scores': [1, 1, 1], 'format': False}, {'id': 50, 'scores': [1, 1, 0], 'format': False}, {'id': 51, 'scores': [1, 1, 1], 'format': False}, {'id': 52, 'scores': [1, 0, 1], 'format': False}, {'id': 53, 'scores': [1, 1, 1], 'format': False}, {'id': 54, 'scores': [1, 1, 1], 'format': False}, {'id': 55, 'scores': [1, 1, 1], 'format': False}, {'id': 56, 'scores': [1, 1, 1], 'format': False}, {'id': 57, 'scores': [1, 1, 1], 'format': False}, {'id': 58, 'scores': [1, 1, 1], 'format': False}, {'id': 59, 'scores': [1, 1, 1], 'format': False}, {'id': 60, 'scores': [1, 1, 1], 'format': False}, {'id': 61, 'scores': None, 'format': None}, {'id': 62, 'scores': [1, 1, 1], 'format': False}, {'id': 63, 'scores': None, 'format': None}, {'id': 64, 'scores': [1, 1, 1], 'format': False}, {'id': 65, 'scores': [1, 1, 1], 'format': False}, {'id': 66, 'scores': [1, 1, 1], 'format': False}, {'id': 67, 'scores': [1, 1, 0], 'format': False}, {'id': 68, 'scores': None, 'format': None}, {'id': 69, 'scores': [1, 1, 1], 'format': False}, {'id': 70, 'scores': [0, 0, 0], 'format': True}, {'id': 71, 'scores': [1, 1, 1], 'format': False}, {'id': 72, 'scores': [1, 1, 1], 'format': False}, {'id': 73, 'scores': [1, 1, 0], 'format': False}, {'id': 74, 'scores': [1, 1, 1], 'format': False}, {'id': 75, 'scores': [1, 1, 1], 'format': False}, {'id': 76, 'scores': None, 'format': None}, {'id': 77, 'scores': None, 'format': None}, {'id': 78, 'scores': [1, 1, 1], 'format': False}, {'id': 79, 'scores': [1, 1, 0], 'format': False}, {'id': 80, 'scores': [1, 1, 1], 'format': False}, {'id': 81, 'scores': [1, 1, 1], 'format': False}, {'id': 82, 'scores': [1, 1, 1], 'format': False}, {'id': 83, 'scores': [1, 1, 1], 'format': False}, {'id': 84, 'scores': [1, 0, 0], 'format': True}, {'id': 85, 'scores': [1, 1, 1], 'format': False}, {'id': 86, 'scores': [1, 1, 1], 'format': False}, {'id': 87, 'scores': [1, 1, 1], 'format': False}, {'id': 88, 'scores': [1, 1, 1], 'format': False}, {'id': 89, 'scores': [1, 1, 1], 'format': False}, {'id': 90, 'scores': [1, 1, 1], 'format': False}, {'id': 91, 'scores': [1, 1, 1], 'format': False}, {'id': 92, 'scores': [1, 1, 1], 'format': False}, {'id': 93, 'scores': [1, 1, 1], 'format': False}, {'id': 94, 'scores': [1, 1, 0], 'format': True}, {'id': 95, 'scores': [1, 1, 1], 'format': False}, {'id': 96, 'scores': [1, 1, 1], 'format': False}, {'id': 97, 'scores': [1, 1, 1], 'format': False}, {'id': 98, 'scores': [1, 1, 0], 'format': False}, {'id': 99, 'scores': None, 'format': None}, {'id': 100, 'scores': [1, 1, 1], 'format': False}, {'id': 101, 'scores': [1, 1, 1], 'format': False}, {'id': 102, 'scores': [1, 0, 0], 'format': False}, {'id': 103, 'scores': [1, 1, 1], 'format': False}, {'id': 104, 'scores': [1, 0, 0], 'format': False}, {'id': 105, 'scores': [1, 1, 1], 'format': False}, {'id': 106, 'scores': [1, 1, 1], 'format': False}, {'id': 107, 'scores': [1, 1, 1], 'format': False}, {'id': 108, 'scores': [1, 0, 1], 'format': False}, {'id': 109, 'scores': [1, 1, 1], 'format': False}, {'id': 110, 'scores': [1, 1, 1], 'format': False}, {'id': 111, 'scores': [1, 1, 1], 'format': False}, {'id': 112, 'scores': [1, 1, 1], 'format': False}, {'id': 113, 'scores': [1, 0, 0], 'format': False}, {'id': 114, 'scores': [1, 1, 1], 'format': False}, {'id': 115, 'scores': [1, 1, 1], 'format': False}, {'id': 116, 'scores': [1, 1, 1], 'format': False}, {'id': 117, 'scores': None, 'format': None}, {'id': 118, 'scores': [1, 1, 1], 'format': False}, {'id': 119, 'scores': [1, 1, 1], 'format': False}, {'id': 120, 'scores': [1, 1, 1], 'format': False}, {'id': 121, 'scores': [1, 1, 1], 'format': False}, {'id': 122, 'scores': [1, 0, 1], 'format': True}, {'id': 123, 'scores': [1, 1, 1], 'format': False}, {'id': 124, 'scores': [1, 1, 0], 'format': False}, {'id': 125, 'scores': [1, 1, 1], 'format': False}, {'id': 126, 'scores': [1, 0, 1], 'format': False}, {'id': 127, 'scores': [1, 1, 1], 'format': False}, {'id': 128, 'scores': [1, 1, 1], 'format': False}, {'id': 129, 'scores': [0, 0, 0], 'format': False}, {'id': 130, 'scores': [1, 1, 1], 'format': False}, {'id': 131, 'scores': [0, 0, 1], 'format': False}, {'id': 132, 'scores': [1, 1, 1], 'format': False}, {'id': 133, 'scores': [0, 1, 1], 'format': False}, {'id': 134, 'scores': [1, 1, 1], 'format': False}, {'id': 135, 'scores': [1, 1, 1], 'format': False}, {'id': 136, 'scores': None, 'format': None}, {'id': 137, 'scores': [1, 1, 1], 'format': False}, {'id': 138, 'scores': [1, 0, 1], 'format': False}, {'id': 139, 'scores': [1, 1, 1], 'format': False}, {'id': 140, 'scores': [1, 1, 1], 'format': False}, {'id': 141, 'scores': [1, 1, 1], 'format': False}, {'id': 142, 'scores': [1, 1, 1], 'format': False}, {'id': 143, 'scores': [0, 0, 1], 'format': False}, {'id': 144, 'scores': [1, 1, 0], 'format': False}, {'id': 145, 'scores': [1, 1, 1], 'format': False}, {'id': 146, 'scores': [0, 0, 0], 'format': False}, {'id': 147, 'scores': None, 'format': None}, {'id': 148, 'scores': [1, 1, 1], 'format': False}, {'id': 149, 'scores': [1, 1, 0], 'format': False}, {'id': 150, 'scores': [1, 1, 1], 'format': False}, {'id': 151, 'scores': [1, 1, 1], 'format': False}, {'id': 152, 'scores': [1, 1, 1], 'format': False}, {'id': 153, 'scores': [1, 1, 1], 'format': False}, {'id': 154, 'scores': [1, 1, 1], 'format': False}, {'id': 155, 'scores': [1, 1, 1], 'format': False}, {'id': 156, 'scores': [1, 1, 1], 'format': False}, {'id': 157, 'scores': [1, 1, 1], 'format': False}, {'id': 158, 'scores': None, 'format': None}, {'id': 159, 'scores': [1, 1, 0], 'format': False}, {'id': 160, 'scores': [0, 1, 0], 'format': True}, {'id': 161, 'scores': [0, 0, 0], 'format': True}, {'id': 162, 'scores': [1, 1, 1], 'format': False}, {'id': 163, 'scores': [1, 1, 1], 'format': False}, {'id': 164, 'scores': [1, 1, 1], 'format': False}, {'id': 165, 'scores': [1, 1, 1], 'format': False}, {'id': 166, 'scores': [1, 1, 1], 'format': False}, {'id': 167, 'scores': [1, 1, 1], 'format': False}, {'id': 168, 'scores': [1, 1, 1], 'format': False}, {'id': 169, 'scores': [1, 1, 1], 'format': False}, {'id': 170, 'scores': [1, 1, 1], 'format': False}, {'id': 171, 'scores': None, 'format': None}, {'id': 172, 'scores': [1, 1, 1], 'format': False}, {'id': 173, 'scores': [1, 1, 1], 'format': False}, {'id': 174, 'scores': [1, 1, 1], 'format': False}, {'id': 175, 'scores': [1, 0, 1], 'format': False}]}]

In [ ]:
for run in RUNS:
    mark = next(row for row in run["marks"] if row["id"] == 10)
    print(f"{run['model']} | condition, geometry, trade = {mark['scores']} | "
          f"{sum(mark['scores'])}/3 | format violation {mark['format']}")

## 4. Read the report

Two answers can earn the same total for different reasoning. On the exchange question the judge awarded every candidate two out of three and accepted the format of all three. The criterion-level marks show which claim the judge credited. A second grader could independently rescore a sample with model names hidden, record the evidence for each mark and investigate disagreements.

The three archived runs each attempted 175 questions worth 525 points. The historical runner could assign zero points after a failed judge call. To make the comparisons below, the public extract marks missing or erroneous responses or judgments, nonbinary marks and inconsistent totals as unscored. First count how many items each run can contribute. Then compare the subset with usable marks for all three. Some excluded questions could be unusually difficult, so this subset describes a selected set of tasks.

In [ ]:
from collections import Counter

by_id = {item["id"]: item for item in BANK}
accepted = [
    {row["id"] for row in run["marks"] if row["scores"] is not None}
    for run in RUNS
]
matched = set.intersection(*accepted)

def summarize(run, item_ids):
    rows = [row for row in run["marks"] if row["id"] in item_ids and row["scores"] is not None]
    earned = sum(sum(row["scores"]) for row in rows)
    possible = sum(by_id[row["id"]]["max_points"] for row in rows)
    flags = sum(row["format"] for row in rows)
    return len(rows), earned, possible, flags

print(f"Items usable for all three runs: {len(matched)} of {len(BANK)}")
print("Topic counts in the full bank:", dict(Counter(item["topic"] for item in BANK)))
for run, valid in zip(RUNS, accepted):
    n, earned, possible, flags = summarize(run, matched)
    print(f"{run['model']} | {len(valid)} usable of {len(BANK)} | "
          f"{earned}/{possible} ({earned/possible:.1%}) on {n} shared items | {flags} format flags")

**Try a different comparison.** Set the topic below to one of the printed topic labels. Check the denominator before interpreting a percentage. A topic with one usable item cannot support the same precision as a topic with dozens. Scores here describe specific model versions responding to a particular bank in January and February 2026.

To adapt the exercise to a RAG system, replace the exchange task with a question about the readings and write one rubric row each for factual accuracy, relevance and support from a cited passage. Give each system the same questions and reading collection. Give graders the evidence needed to verify citations. Report disagreements on those three dimensions separately so that a plausible answer with a weak citation is visible.

In [ ]:
topic = "ge_exchange"
known_topics = {item["topic"] for item in BANK}
if topic not in known_topics:
    raise ValueError(f"Choose one of the available topics: {sorted(known_topics)}")

topic_ids = {item_id for item_id in matched if by_id[item_id]["topic"] == topic}
for run in RUNS:
    n, earned, possible, flags = summarize(run, topic_ids)
    print(f"{topic} | {run['model']} | {n} shared items | {earned}/{possible} points | {flags} format flags")